# VAE / CVAE CelebA - Notebook Google Colab

Notebook autonome pour entrainer un VAE et un CVAE sur CelebA avec un sous-echantillon equilibre par attributs.

Attributs utilises: `Smiling`, `Male`, `Wavy_Hair`.

Objectifs:
- charger CelebA depuis Hugging Face (`tpremoli/CelebA-attrs`) ;
- construire un sous-echantillon equilibre par combinaison d'attributs ;
- entrainer VAE et CVAE ;
- sauvegarder `best_checkpoint.pth`, `last_checkpoint.pth`, logs CSV et figures ;
- logger les runs dans MLflow localement si souhaite.

## 1. Installation

Sur Colab, commence par activer un runtime GPU: `Runtime > Change runtime type > T4 GPU`.

In [ ]:
!pip -q install datasets pillow matplotlib scikit-learn tqdm pyyaml mlflow


In [ ]:
import csv
import itertools
import json
import math
import os
import random
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.utils import make_grid, save_image
from tqdm.auto import tqdm

try:
    import mlflow
except Exception:
    mlflow = None

print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())


## 2. Configuration

Les valeurs ci-dessous reprennent le protocole actuel du projet: 32k images pour VAE/CVAE, 100 epochs, latent 128, largeur 64, beta 0.5 avec KL annealing.

Important: `epochs=100` est la duree maximale totale. `warmup_epochs=10` veut seulement dire que beta monte progressivement pendant les 10 premieres epochs, puis reste a 0.5. L'early stopping ne commence a compter la patience qu'apres l'epoch 20 pour eviter un arret trop tot pendant la stabilisation.

Par defaut, les resultats sont sauvegardes dans `/content/celeba_runs`. Si tu veux les garder apres la fermeture du runtime Colab, mets `USE_GOOGLE_DRIVE = True` dans la cellule suivante.


In [ ]:
CONFIG = {
    'dataset': {
        'name': 'celeba',
        'hf_name': 'tpremoli/CelebA-attrs',
        'image_size': (64, 64),
        'channels': 3,
        'attributes': ['Smiling', 'Male', 'Wavy_Hair'],
        'sampling_strategy': 'balanced_conditions',
        'seed': 42,
        'n_train': 32000,
        'n_val': 3000,
        'n_test': 3000,
    },
    'model': {
        'latent_dim': 128,
        'hidden_channels': 64,
    },
    'training': {
        'batch_size': 64,
        'epochs': 100,
        'lr': 1e-3,
        'beta': 0.5,
        'early_stopping': {'enabled': True, 'patience': 10, 'min_delta': 1.0, 'start_after_epoch': 20},
        'kl_annealing': {'enabled': True, 'start_beta': 0.0, 'final_beta': 0.5, 'warmup_epochs': 10},
    },
    'mlflow': {
        'enabled': True,
        'tracking_uri': 'sqlite:////content/mlflow.db',
        'experiment_name': 'blaise_celeba_colab',
    }
}

USE_GOOGLE_DRIVE = False

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT = Path('/content/drive/MyDrive/celeba_runs')
    CACHE_DIR = Path('/content/drive/MyDrive/celeba_cache')
else:
    OUTPUT_ROOT = Path('/content/celeba_runs')
    CACHE_DIR = Path('/content/celeba_cache')

FIGURE_DIR = OUTPUT_ROOT / 'figures'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG['dataset']['seed'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


## 3. Chargement CelebA et sampling equilibre

La commande d'inspection naturelle montre le desequilibre du dataset. Le loader d'entrainement, lui, utilise `balanced_conditions`: environ 1/8 des images par combinaison des 3 attributs.

In [ ]:
HF_SPLITS = {'train': 'train', 'val': 'validation', 'test': 'test'}

def condition_key(row, attributes):
    return tuple(1 if row[name] > 0 else 0 for name in attributes)

def balanced_condition_indices(hf_dataset, n_samples, seed, attributes):
    rng = np.random.default_rng(seed)
    attr_dataset = hf_dataset.select_columns(list(attributes))
    groups = {}
    for idx, row in enumerate(attr_dataset):
        groups.setdefault(condition_key(row, attributes), []).append(idx)
    for indices in groups.values():
        rng.shuffle(indices)

    n_take = min(n_samples, len(hf_dataset))
    combos = sorted(groups)
    min_group_size = min(len(groups[c]) for c in combos)
    base = min(n_take // len(combos), min_group_size)

    selected = []
    offsets = {}
    for combo in combos:
        selected.extend(groups[combo][:base])
        offsets[combo] = base

    remaining = n_take - len(selected)
    while remaining > 0:
        progress = False
        order = list(combos)
        rng.shuffle(order)
        for combo in order:
            if remaining <= 0:
                break
            offset = offsets[combo]
            if offset >= len(groups[combo]):
                continue
            selected.append(groups[combo][offset])
            offsets[combo] += 1
            remaining -= 1
            progress = True
        if not progress:
            break

    rng.shuffle(selected)
    print('Balanced counts:', {combo: offsets[combo] for combo in combos})
    return selected

def cache_path(split, n_samples, seed, image_size, attributes, sampling_strategy):
    attrs_tag = '-'.join(attributes)
    return CACHE_DIR / f'{split}_n{n_samples}_seed{seed}_{sampling_strategy}_{image_size[0]}x{image_size[1]}_{attrs_tag}.npz'

def load_split(split, n_samples, config):
    ds_cfg = config['dataset']
    image_size = tuple(ds_cfg['image_size'])
    attributes = ds_cfg['attributes']
    seed = ds_cfg['seed']
    strategy = ds_cfg.get('sampling_strategy', 'balanced_conditions')
    path = cache_path(split, n_samples, seed, image_size, attributes, strategy)
    if path.exists():
        with np.load(path) as data:
            return data['images'], data['attributes']

    print(f'Loading HF split {HF_SPLITS[split]}...')
    hf_dataset = load_dataset(ds_cfg['hf_name'], split=HF_SPLITS[split])
    if strategy == 'balanced_conditions':
        indices = balanced_condition_indices(hf_dataset, n_samples, seed, attributes)
        sampled = hf_dataset.select(indices)
    else:
        sampled = hf_dataset.shuffle(seed=seed).select(range(min(n_samples, len(hf_dataset))))

    images = np.zeros((len(sampled), image_size[0], image_size[1], 3), dtype=np.uint8)
    attrs = np.zeros((len(sampled), len(attributes)), dtype=np.float32)
    for i, row in enumerate(tqdm(sampled, desc=f'prepare {split}')):
        image = row['image'].convert('RGB').resize(image_size, Image.BILINEAR)
        images[i] = np.asarray(image, dtype=np.uint8)
        for j, name in enumerate(attributes):
            attrs[i, j] = 1.0 if row[name] > 0 else 0.0

    np.savez_compressed(path, images=images, attributes=attrs)
    print('Saved cache:', path)
    return images, attrs

class CelebASubsample(Dataset):
    def __init__(self, images, attrs):
        self.images = images
        self.attrs = attrs
    def __len__(self):
        return len(self.images)
    def __getitem__(self, idx):
        x = self.images[idx].astype(np.float32) / 255.0
        x = x * 2.0 - 1.0
        x = torch.from_numpy(x).permute(2, 0, 1).contiguous()
        c = torch.from_numpy(self.attrs[idx])
        return x, c

def build_dataloaders(config):
    ds_cfg = config['dataset']
    train = CelebASubsample(*load_split('train', ds_cfg['n_train'], config))
    val = CelebASubsample(*load_split('val', ds_cfg['n_val'], config))
    test = CelebASubsample(*load_split('test', ds_cfg['n_test'], config))
    batch_size = config['training']['batch_size']
    return (
        DataLoader(train, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available()),
        DataLoader(val, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available()),
        DataLoader(test, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available()),
    )

def print_combo_distribution(loader, attributes):
    counts = Counter()
    total = 0
    for _, attrs in loader:
        for row in attrs:
            key = tuple(int(v.item() > 0.5) for v in row)
            counts[key] += 1
            total += 1
    for key in sorted(counts):
        label = ' '.join(f'{name}={value}' for name, value in zip(attributes, key))
        print(f'{label:38s} {counts[key]:6d} ({100*counts[key]/total:5.1f}%)')


In [ ]:
# Charge les donnees. Le premier passage peut prendre du temps.
train_loader, val_loader, test_loader = build_dataloaders(CONFIG)
print('Train distribution:')
print_combo_distribution(train_loader, CONFIG['dataset']['attributes'])


## 4. Modeles VAE et CVAE

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.LeakyReLU(0.2, inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class ConvTransposeBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class VAE(nn.Module):
    def __init__(self, channels=3, latent_dim=128, hidden_channels=64):
        super().__init__()
        self.latent_dim = latent_dim
        c1, c2, c3, c4 = hidden_channels, hidden_channels*2, hidden_channels*4, hidden_channels*8
        self.encoder = nn.Sequential(ConvBlock(channels, c1), ConvBlock(c1, c2), ConvBlock(c2, c3), ConvBlock(c3, c4), nn.Flatten())
        self.encoder_output_dim = c4 * 4 * 4
        self.fc_mu = nn.Linear(self.encoder_output_dim, latent_dim)
        self.fc_logvar = nn.Linear(self.encoder_output_dim, latent_dim)
        self.decoder_input = nn.Linear(latent_dim, self.encoder_output_dim)
        self.decoder = nn.Sequential(
            nn.Unflatten(1, (c4, 4, 4)),
            ConvTransposeBlock(c4, c3), ConvTransposeBlock(c3, c2), ConvTransposeBlock(c2, c1),
            nn.ConvTranspose2d(c1, channels, kernel_size=4, stride=2, padding=1),
            nn.Tanh(),
        )
    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    def decode(self, z):
        return self.decoder(self.decoder_input(z))
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar
    @torch.no_grad()
    def sample(self, n):
        z = torch.randn(n, self.latent_dim, device=next(self.parameters()).device)
        return self.decode(z)

class CVAE(nn.Module):
    def __init__(self, channels=3, latent_dim=128, hidden_channels=64, num_conditions=3):
        super().__init__()
        self.latent_dim = latent_dim
        self.num_conditions = num_conditions
        c1, c2, c3, c4 = hidden_channels, hidden_channels*2, hidden_channels*4, hidden_channels*8
        self.encoder = nn.Sequential(ConvBlock(channels, c1), ConvBlock(c1, c2), ConvBlock(c2, c3), ConvBlock(c3, c4), nn.Flatten())
        self.encoder_output_dim = c4 * 4 * 4
        self.fc_mu = nn.Linear(self.encoder_output_dim + num_conditions, latent_dim)
        self.fc_logvar = nn.Linear(self.encoder_output_dim + num_conditions, latent_dim)
        self.decoder_input = nn.Linear(latent_dim + num_conditions, self.encoder_output_dim)
        self.decoder = nn.Sequential(
            nn.Unflatten(1, (c4, 4, 4)),
            ConvTransposeBlock(c4, c3), ConvTransposeBlock(c3, c2), ConvTransposeBlock(c2, c1),
            nn.ConvTranspose2d(c1, channels, kernel_size=4, stride=2, padding=1),
            nn.Tanh(),
        )
    def encode(self, x, c):
        h = self.encoder(x)
        h = torch.cat([h, c], dim=1)
        return self.fc_mu(h), self.fc_logvar(h)
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    def decode(self, z, c):
        return self.decoder(self.decoder_input(torch.cat([z, c], dim=1)))
    def forward(self, x, c):
        mu, logvar = self.encode(x, c)
        z = self.reparameterize(mu, logvar)
        return self.decode(z, c), mu, logvar
    @torch.no_grad()
    def sample(self, c, n=1):
        if c.dim() == 1:
            c = c.unsqueeze(0).expand(n, -1)
        c = c.to(next(self.parameters()).device).float()
        z = torch.randn(c.size(0), self.latent_dim, device=c.device)
        return self.decode(z, c)

def elbo_loss(x_hat, x, mu, logvar, beta=1.0):
    recon = F.mse_loss(x_hat, x, reduction='sum') / x.size(0)
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
    return {'loss': recon + beta * kl, 'reconstruction': recon, 'kl': kl, 'beta': beta}

def beta_for_epoch(config, epoch):
    tr = config['training']
    final_beta = float(tr.get('beta', 1.0))
    ann = tr.get('kl_annealing', {})
    if not ann.get('enabled', False):
        return final_beta
    start = float(ann.get('start_beta', 0.0))
    final = float(ann.get('final_beta', final_beta))
    warmup = max(1, int(ann.get('warmup_epochs', 10)))
    progress = min(1.0, epoch / warmup)
    return start + progress * (final - start)


## 5. Entrainement avec checkpoints et MLflow

In [ ]:
def save_checkpoint(path, model, optimizer, epoch, best_loss, best_epoch, config):
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'epoch': epoch,
        'best_loss': best_loss,
        'best_epoch': best_epoch,
        'configuration': config,
    }, path)

def run_epoch(model, loader, optimizer, conditioned, beta):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total = {'loss': 0.0, 'reconstruction': 0.0, 'kl': 0.0}
    n = 0
    for x, attrs in tqdm(loader, leave=False):
        x = x.to(device, non_blocking=True)
        attrs = attrs.to(device, non_blocking=True).float()
        if is_train:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(is_train):
            if conditioned:
                x_hat, mu, logvar = model(x, attrs)
            else:
                x_hat, mu, logvar = model(x)
            metrics = elbo_loss(x_hat, x, mu, logvar, beta)
            if is_train:
                metrics['loss'].backward()
                optimizer.step()
        bs = x.size(0)
        for k in total:
            total[k] += metrics[k].item() * bs
        n += bs
    return {k: v / n for k, v in total.items()} | {'beta': beta}

def train_model(model, model_name, conditioned=False, config=CONFIG):
    output_dir = OUTPUT_ROOT / model_name
    output_dir.mkdir(parents=True, exist_ok=True)
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config['training']['lr'])
    csv_path = output_dir / 'training_log.csv'
    best_loss = float('inf')
    best_epoch = 0
    patience = config['training']['early_stopping']['patience']
    min_delta = config['training']['early_stopping']['min_delta']
    start_after_epoch = config['training']['early_stopping'].get('start_after_epoch', 0)
    no_improve = 0

    use_mlflow = config['mlflow'].get('enabled', False) and mlflow is not None
    if use_mlflow:
        mlflow.set_tracking_uri(config['mlflow']['tracking_uri'])
        mlflow.set_experiment(config['mlflow']['experiment_name'])
        run_ctx = mlflow.start_run(run_name=model_name)
    else:
        run_ctx = None

    try:
        if use_mlflow:
            mlflow.log_params({
                'model_name': model_name,
                'conditioned': conditioned,
                'latent_dim': config['model']['latent_dim'],
                'hidden_channels': config['model']['hidden_channels'],
                'n_train': config['dataset']['n_train'],
                'n_val': config['dataset']['n_val'],
                'sampling_strategy': config['dataset']['sampling_strategy'],
                'epochs': config['training']['epochs'],
                'lr': config['training']['lr'],
                'beta': config['training']['beta'],
                'early_stopping_start_after_epoch': start_after_epoch,
            })

        with open(csv_path, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=['epoch', 'phase', 'loss', 'reconstruction', 'kl', 'beta'])
            writer.writeheader()

        for epoch in range(1, config['training']['epochs'] + 1):
            beta = beta_for_epoch(config, epoch)
            train_m = run_epoch(model, train_loader, optimizer, conditioned, beta)
            val_m = run_epoch(model, val_loader, None, conditioned, beta)
            print(f"{model_name} epoch {epoch}: val_loss={val_m['loss']:.2f}, recon={val_m['reconstruction']:.2f}, kl={val_m['kl']:.2f}, beta={beta:.3f}")

            with open(csv_path, 'a', newline='') as f:
                writer = csv.DictWriter(f, fieldnames=['epoch', 'phase', 'loss', 'reconstruction', 'kl', 'beta'])
                writer.writerow({'epoch': epoch, 'phase': 'train', **train_m})
                writer.writerow({'epoch': epoch, 'phase': 'val', **val_m})

            if use_mlflow:
                mlflow.log_metrics({f'train_{k}': float(v) for k, v in train_m.items()}, step=epoch)
                mlflow.log_metrics({f'val_{k}': float(v) for k, v in val_m.items()}, step=epoch)

            improved = val_m['loss'] < best_loss - min_delta
            if improved:
                best_loss = val_m['loss']
                best_epoch = epoch
                no_improve = 0
                save_checkpoint(output_dir / 'best_checkpoint.pth', model, optimizer, epoch, best_loss, best_epoch, config)
            elif epoch > start_after_epoch:
                no_improve += 1

            save_checkpoint(output_dir / 'last_checkpoint.pth', model, optimizer, epoch, best_loss, best_epoch, config)
            if config['training']['early_stopping']['enabled'] and epoch > start_after_epoch and no_improve >= patience:
                print(f'Early stopping apres epoch {start_after_epoch}.')
                break

        if use_mlflow:
            mlflow.log_metrics({'best_val_loss': best_loss, 'best_epoch': best_epoch})
            mlflow.log_artifact(str(csv_path), artifact_path='logs')
            for ckpt in ['best_checkpoint.pth', 'last_checkpoint.pth']:
                mlflow.log_artifact(str(output_dir / ckpt), artifact_path='checkpoints')
    finally:
        if use_mlflow:
            mlflow.end_run()
    return model, output_dir


## 6. Lancer l'entrainement

Tu peux lancer VAE, CVAE, ou les deux. Sur Colab GPU, 32k images en 100 epochs peut quand meme prendre du temps.

In [ ]:
vae = VAE(latent_dim=CONFIG['model']['latent_dim'], hidden_channels=CONFIG['model']['hidden_channels'])
vae, vae_dir = train_model(vae, 'vae_improved_colab', conditioned=False)


In [ ]:
cvae = CVAE(latent_dim=CONFIG['model']['latent_dim'], hidden_channels=CONFIG['model']['hidden_channels'], num_conditions=len(CONFIG['dataset']['attributes']))
cvae, cvae_dir = train_model(cvae, 'cvae_improved_colab', conditioned=True)


## 7. Generer des grilles d'images

In [ ]:
@torch.no_grad()
def generate_vae_grids(model, output_dir, n=8):
    model.eval()
    x, _ = next(iter(test_loader))
    x = x[:n].to(device)
    mu, _ = model.encode(x)
    x_hat = model.decode(mu)
    grid = make_grid(torch.cat([x.cpu(), x_hat.cpu()], dim=0), nrow=n, normalize=True, value_range=(-1, 1))
    path = output_dir / 'vae_reconstruction_grid.png'
    save_image(grid, path)
    z_samples = model.sample(n*n).cpu()
    sample_grid = make_grid(z_samples, nrow=n, normalize=True, value_range=(-1, 1))
    sample_path = output_dir / 'vae_random_samples_grid.png'
    save_image(sample_grid, sample_path)
    print(path, sample_path)
    return path, sample_path

@torch.no_grad()
def generate_cvae_grid(model, output_dir, samples_per_combo=8):
    model.eval()
    rows = []
    for combo in itertools.product([0.0, 1.0], repeat=len(CONFIG['dataset']['attributes'])):
        c = torch.tensor(combo, dtype=torch.float32, device=device)
        rows.append(model.sample(c, n=samples_per_combo).cpu())
    all_samples = torch.cat(rows, dim=0)
    grid = make_grid(all_samples, nrow=samples_per_combo, normalize=True, value_range=(-1, 1))
    path = output_dir / 'cvae_grid.png'
    save_image(grid, path)
    print(path)
    return path

vae_figs = generate_vae_grids(vae, FIGURE_DIR)
cvae_fig = generate_cvae_grid(cvae, FIGURE_DIR)


In [ ]:
from IPython.display import Image as IPImage, display
for path in list(vae_figs) + [cvae_fig]:
    display(IPImage(filename=str(path)))


## 8. Evaluation quantitative simple

In [ ]:
@torch.no_grad()
def evaluate_recon_kl(model, loader, conditioned=False):
    model.eval()
    total_recon, total_kl, total = 0.0, 0.0, 0
    for x, attrs in tqdm(loader):
        x = x.to(device)
        attrs = attrs.to(device).float()
        if conditioned:
            x_hat, mu, logvar = model(x, attrs)
        else:
            x_hat, mu, logvar = model(x)
        m = elbo_loss(x_hat, x, mu, logvar, beta=CONFIG['training']['beta'])
        bs = x.size(0)
        total_recon += m['reconstruction'].item() * bs
        total_kl += m['kl'].item() * bs
        total += bs
    return {'reconstruction': total_recon / total, 'kl': total_kl / total, 'n_test': total}

results = {
    'vae': evaluate_recon_kl(vae, test_loader, conditioned=False),
    'cvae': evaluate_recon_kl(cvae, test_loader, conditioned=True),
}
print(json.dumps(results, indent=2))
with open(OUTPUT_ROOT / 'comparison.json', 'w') as f:
    json.dump(results, f, indent=2)


## 9. Recuperer les fichiers

Les sorties sont dans `OUTPUT_ROOT`. Par defaut, cela correspond a `/content/celeba_runs`. Si `USE_GOOGLE_DRIVE=True`, elles sont sauvegardees dans `MyDrive/celeba_runs`.

Fichiers importants:
- `vae_improved_colab/best_checkpoint.pth`
- `cvae_improved_colab/best_checkpoint.pth`
- `figures/*.png`
- `comparison.json`

Tu peux les telecharger depuis le panneau de fichiers Colab, les conserver dans Drive, ou zipper le dossier.


In [ ]:
import shutil
zip_base = shutil.make_archive('/content/celeba_runs', 'zip', root_dir=str(OUTPUT_ROOT))
print(zip_base)


## 10. Interface MLflow dans Colab

Colab ne donne pas toujours un acces direct au port local. Le plus simple est de telecharger `mlflow.db` et les artefacts, ou d'utiliser `pyngrok` si tu veux exposer l'UI. Pour lancer l'UI localement dans Colab:

In [ ]:
# Optionnel: lance l'UI MLflow dans le runtime Colab.
# !mlflow ui --backend-store-uri sqlite:////content/mlflow.db --host 0.0.0.0 --port 5000
